In [1]:
import os
import json
import pandas as pd
from pathlib import Path

In [33]:
def extract_model_metadata(root_dir):
    data_list = []
    root_path = Path(root_dir)

    # Iterate through all subdirectories
    for subdir in root_path.iterdir():
        if subdir.is_dir():
            json_path = subdir / "metadata.json"
            
            if json_path.exists():
                with open(json_path, 'r') as f:
                    meta = json.load(f)
                
                # Flatten the specific fields into a dictionary
                row = {
                    'model_name': subdir.name,
                    'version': meta.get('version'),
                    'data_snapshot': meta.get('data_snapshot'),
                    'model_type': meta.get('config', {}).get('model_type'),
                    'train_rows': meta.get('training_data', {}).get('total_train_rows'),
                    'test_rows': meta.get('training_data', {}).get('test_rows'),
                    'classes_test': meta.get('training_data', {}).get('target_balance_test'),
                    'classes_train': meta.get('training_data', {}).get('target_balance_train'),
                    'roc_auc': meta.get('result', {}).get('roc_auc'),
                    'accuracy': meta.get('result', {}).get('accuracy'),
                    'precision_above': meta.get('result', {}).get('precision_above'),
                    'recall_above': meta.get('result', {}).get('recall_above'),
                    'f1_above': meta.get('result', {}).get('f1_above'),
                    'precision_below': meta.get('result', {}).get('precision_below'),
                    'recall_below': meta.get('result', {}).get('recall_below'),
                    'f1_below': meta.get('result', {}).get('f1_below'),
                    'confusion_matrix': meta.get('result', {}).get('confusion_matrix'),
                    'features_count': meta.get('feature_count'),
                    'features': meta.get('features'),
                    'top_features': meta.get('result', {}).get('top_features')
                }
                data_list.append(row)
    df_ = pd.DataFrame(data_list).sort_values('model_name').reset_index(drop=True)
    df_.insert(2, 'generation', df_['version'].str.extract(r'(v\d+\.\d+)'))
    #re.match(r'v\d+\.\d+', df_['version']).group(),
    return df_

# Usage: 
# df = extract_model_metadata('./models')
# print(df.head())

In [34]:
models_df = extract_model_metadata("../../../data/models")
print(models_df.head(10))
print(models_df.columns)

   model_name     version generation       data_snapshot          model_type  \
0  v1.0_lr_l1  v1.0_lr_l1       v1.0  v1.0_model_mixed66  LogisticRegression   
1     v1.0_rf     v1.0_rf       v1.0  v1.0_model_mixed66        RandomForest   
2    v1.0_xgb    v1.0_xgb       v1.0  v1.0_model_mixed66             XGBoost   
3  v2.0_lr_l1  v2.0_lr_l1       v2.0   v2.0_mixed_70real  LogisticRegression   
4     v2.0_rf     v2.0_rf       v2.0   v2.0_mixed_70real        RandomForest   
5    v2.0_xgb    v2.0_xgb       v2.0   v2.0_mixed_70real             XGBoost   
6  v3.0_lr_l1  v3.0_lr_l1       v3.0   v3.0_mixed_80real  LogisticRegression   
7     v3.0_rf     v3.0_rf       v3.0   v3.0_mixed_80real        RandomForest   
8    v3.0_xgb    v3.0_xgb       v3.0   v3.0_mixed_80real             XGBoost   
9  v3.1_lr_l1  v3.1_lr_l1       v3.1   v3.1_mixed_80real  LogisticRegression   

   train_rows  test_rows          classes_test           classes_train  \
0        7100        525  {'1': 297, '0': 228

In [ ]:
# Uncomment to write the results out
# models_df.to_csv('model_results_df.csv', index=False)